# Phase 3: 상권 데이터(POI) 버퍼 피처 엔지니어링 프로토타입
이 노트북은 전체 데이터를 다루기 전, 작은 스케일(Sample)의 데이터를 통해 30m, 50m, 100m 다중 버퍼 sjoin 연산이 올바르게 카운트 되는지 확인하기 위한 실험 환경입니다.

In [12]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from pathlib import Path

# -- 설정 --
PROJ_CRS = "EPSG:5179"  # 10m 단위 계산용 투영 좌표계 (KATEC)
BASE_CRS = "EPSG:4326"  # 원본 CSV 위경도 좌표계

print("✅ 라이브러리 및 공간 설정 세팅 완료!")

✅ 라이브러리 및 공간 설정 세팅 완료!


In [13]:
# 1. 원본 POI 데이터 로드 및 최적화 (스케일 다운)
# 전체 7만건을 연산하기 전, 상위 500개만 가져와서 테스트합니다.
poi_df = pd.read_csv('../../data/raw/poi_gwangju_raw.csv')
print(f"📌 [Sample] POI 데이터 로드 완료: {poi_df.shape}")

# 위경도 결측치가 있는 데이터 제거
poi_df = poi_df.dropna(subset=['경도', '위도'])

# Point 지오메트리화 및 GeoDataFrame 변환
geometry = [Point(xy) for xy in zip(poi_df['경도'], poi_df['위도'])]
poi_gdf = gpd.GeoDataFrame(poi_df, geometry=geometry, crs=BASE_CRS)

# [Rule 1 적용]: Sjoin 및 버퍼(m단위)를 위해 투영 좌표계(EPSG:5179)로 강제 변환
poi_proj = poi_gdf.to_crs(PROJ_CRS)
print("✅ EPSG:5179 좌표계 투영 완료!")
display(poi_proj.head(2))

📌 [Sample] POI 데이터 로드 완료: (75025, 39)
✅ EPSG:5179 좌표계 투영 완료!


,상가업소번호,상호명,지점명,상권업종대분류코드,상권업종대분류명,상권업종중분류코드,상권업종중분류명,상권업종소분류코드,상권업종소분류명,표준산업분류코드,...,건물명,도로명주소,구우편번호,신우편번호,동정보,층정보,호정보,경도,위도,geometry
0,MA010120220813528920,정비대장,NaN,S2,수리·개인,S203,자동차 수리·세차,S20301,자동차 정비소,S95212,...,NaN,광주광역시 광산구 목련로 371,506307,62303,NaN,NaN,NaN,126.832049,35.181646,POINT (939182.89 1687576.441)
1,MA010120220809992707,모다,NaN,G2,소매,G209,섬유·의복·신발 소매,G20910,신발 소매업,G47430,...,NaN,광주광역시 서구 월드컵4강로 75,502839,62026,NaN,NaN,NaN,126.873903,35.143560,POINT (942967.217 1683327.889)


In [ ]:
# 2-1. 상권 카테고리화 및 분할 (누락 없는 동적 매핑 로직)
def categorize_poi(row):
    large = str(row['상권업종대분류명'])
    med = str(row['상권업종중분류명']).strip()
    
    if med == '주점':
        return 'nightlife'
    elif med in ['비알코올', '기타 간이']:
        return 'cafe'
    elif large == '음식': # 주점, 카페를 제외한 나머지 모든 음식(한식, 서양식, 동남아시아 등) 자동 포함
        return 'food'
    elif large == '소매':
        return 'retail'
    else:
        return 'service'
        
poi_proj['category'] = poi_proj.apply(categorize_poi, axis=1)

# 카테고리 분할 및 데이터 유실(Data Loss) 검증
categories = ['nightlife', 'cafe', 'food', 'retail', 'service']
poi_dict = {cat: poi_proj[poi_proj['category'] == cat] for cat in categories}
total_categorized = sum(len(df) for df in poi_dict.values())

print(f"✅ 카테고리별 POI 분할 완료 (원본 {len(poi_proj):,}개 == 분류완료 {total_categorized:,}개):")
for cat, df in poi_dict.items():
    print(f"  - {cat}: {len(df):,}개")

if total_categorized != len(poi_proj):
    print("🔥 경고: 누락된 데이터가 있습니다! 매핑 로직을 확인하세요.")
else:
    print("✨ 누락된 데이터 없이 100% 완벽하게 매핑되었습니다.")

In [ ]:
# 2-2. 타겟 격자(Grid) 도화지 로드
processed_dir = Path('../../data/processed')
gpkg_files = list(processed_dir.glob('*.gpkg'))

if not gpkg_files:
    print("🔥 경고: 아직 생성된 격자 데이터(.gpkg)가 존재하지 않습니다. 먼저 grid_maker.py를 실행해주세요!")
else:
    gpkg_path = gpkg_files[-1]
    grid_masked = gpd.read_file(gpkg_path)
    print(f"🔥 타겟 Grid [{gpkg_path.name}] 전체 로딩 완료: {grid_masked.shape}")
    
    grid_centers = grid_masked.copy()
    grid_centers.geometry = grid_masked.centroid

In [ ]:
# 3. 다중 버퍼 Sjoin 엔진 (Single Responsibility: 공간 조인 연산)
import time
if gpkg_files:
    t0 = time.time()
    print("✂️ 다중 버퍼 sjoin 연산 시작...")
    
    for cat, poi_subset in poi_dict.items():
        if len(poi_subset) == 0: continue
            
        for r in [30, 50, 100]:
            buffer_gdf = grid_centers.copy()
            buffer_gdf.geometry = buffer_gdf.geometry.buffer(r)
            
            # Rule 2: sindex 기반 공간조인
            joined = gpd.sjoin(poi_subset, buffer_gdf, predicate='within')
            
            col_name = f"{cat}_count_{r}m"
            if 'index_right' in joined.columns:
                poi_counts = joined.groupby('index_right').size().rename(col_name)
            else:
                poi_counts = pd.Series(name=col_name, dtype=int)
                
            # .join() 고속 매핑
            grid_masked = grid_masked.join(poi_counts, how='left')
            grid_masked[col_name] = grid_masked[col_name].fillna(0).astype(int)
            
    print(f"⏱️ [종료] 카테고리 기반 15개 피처 연산 성공 (소요시간: {time.time()-t0:.2f}초)")

In [ ]:
# 4. 최종 데이터 검증 및 결과 확인
if gpkg_files:
    print(f"📌 [피처 연산 결과 요약]")
    print(f"전체 분석된 격자(Grid) 수: {len(grid_masked):,}개\n")
    
    # 카테고리별 100m 영향권 커버리지(Coverage) 출력
    for cat in ['nightlife', 'cafe', 'food', 'retail', 'service']:
        col = f"{cat}_count_100m"
        overlap = len(grid_masked[grid_masked[col] > 0])
        print(f"  - 100m 반경 내 [{cat}] 상가가 1개 이상인 격자: {overlap:,}개 ({overlap/len(grid_masked)*100:.1f}%)")
        
    print("\n🔍 [핫스팟 상세 분석]")
    print("유흥(nightlife) 시설 밀집도 최상위 10개 격자 도출:")
    display_columns = ['grid_id', 'geometry', 'nightlife_count_30m', 'cafe_count_30m', 'food_count_30m', 'retail_count_30m']
    display(grid_masked.sort_values(by='nightlife_count_30m', ascending=False)[display_columns].head(10))